# 🛣️ MY-VID Project: Complete Data Processing & Training Pipeline (RO2)

> **Author:** Lim Zi Xuan  
> **Project:** Development of a Standalone and Open-Source Vehicle Detection Framework Using YOLO for Traffic Analysis  

---

## 1. Introduction
This notebook documents the full lifecycle of the MY-VID dataset creation and model training. It fulfills **Research Objective 2 (RO2)** by providing a reproducible workflow for:

1.  **Data Preparation:** Frame extraction, sampling, and privacy blurring.
2.  **Annotation Management:** Semi-automated labeling tools and format conversion (LabelMe $\rightarrow$ YOLO).
3.  **Dataset Construction:** Splitting (Train/Val/Test) and augmentation (Flip/Grayscale).
4.  **Model Training:** Training YOLOv11 on the processed dataset.

---

**Notes before running:**
- Update all path strings to match your local environment or Colab mount paths.
- For Colab training, mount Google Drive or use `gdown` to download dataset and model weights.
- Some cells install packages; only run them once in a fresh environment.

In [ ]:
# Install necessary libraries if missing
# !pip install opencv-python tqdm albumentations screeninfo ultralytics

import os
import cv2
import json
import random
import shutil
import numpy as np
import albumentations as A
from tqdm import tqdm
from ultralytics import YOLO
from datetime import datetime
from matplotlib import pyplot as plt

print("✅ All libraries loaded successfully.")

---
## 2. Pre-Annotation Processing
*(Note: In this stage, the data annotation tools should be done in LabelMe annotation tool. This section focuses on the post-blurring annotation workflow.)*

### 2.1 Semi-Automated Annotation Tool
After blurring, we use a custom Python tool to review and correct annotations. This tool allows for rapid relabeling of JKR classes using keyboard shortcuts (1-6).

**Key Bindings:**
* `1-6`: Assign JKR Class 1-6
* `S`: Skip vehicle
* `A` / `D`: Previous / Next vehicle
* `ESC`: Save & Exit


In [ ]:
# Define JKR Classes and Colors
CLASS_MAP = {
    'Class 1': ('Car/Taxi', (255, 0, 0)),          # Blue
    'Class 2': ('Van/Utility', (255, 255, 0)),     # Cyan
    'Class 3': ('Light Truck', (0, 255, 0)),       # Green
    'Class 4': ('Heavy Truck', (0, 255, 255)),     # Yellow
    'Class 5': ('Bus', (255, 0, 255)),             # Magenta
    'Class 6': ('Motorcycle', (0, 0, 255)),        # Red
    'skipped': ('Skipped', (0, 165, 255))          # Orange
}

KEY_BINDINGS = {
    ord('1'): 'Class 1', ord('2'): 'Class 2', ord('3'): 'Class 3',
    ord('4'): 'Class 4', ord('5'): 'Class 5', ord('6'): 'Class 6',
    ord('s'): 'skipped'
}

def draw_and_relabel(image_path, json_path, folder_name="", image_index=1, total_images=1, max_width=1200, max_height=800):
    image = cv2.imread(image_path)
    if image is None: return

    # Scale image to fit screen
    h, w = image.shape[:2]
    scale = min(max_width / w, max_height / h, 1.0)
    width, height = int(w * scale), int(h * scale)

    with open(json_path, 'r') as f: data = json.load(f)
    shapes = data.get('shapes', [])
    current_index = 0
    window_name = f"Annotation Tool - {folder_name}"

    cv2.namedWindow(window_name, cv2.WINDOW_NORMAL)
    cv2.resizeWindow(window_name, width, height)

    while 0 <= current_index < len(shapes):
        shape = shapes[current_index]
        points = [(int(x), int(y)) for x, y in shape['points']]
        label = shape.get('label', 'unlabeled')
        
        temp_img = image.copy()
        
        # Highlight current vehicle
        if len(points) >= 2:
            cv2.polylines(temp_img, [np.array(points)], True, (0, 255, 0), 5)
            cv2.putText(temp_img, f"Current: {label}", points[0], cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)

        # UI Overlay
        resized = cv2.resize(temp_img, (width, height))
        cv2.putText(resized, f"Vehicle {current_index+1}/{len(shapes)} | [1-6] to Relabel | [A/D] Navigate", (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)
        
        cv2.imshow(window_name, resized)
        key = cv2.waitKey(0)

        if key in KEY_BINDINGS:
            shape['label'] = KEY_BINDINGS[key]
            current_index += 1
        elif key == ord('d'): current_index += 1
        elif key == ord('a'): current_index -= 1
        elif key == 27: break # ESC

    with open(json_path, 'w') as f: json.dump(data, f, indent=4)
    cv2.destroyWindow(window_name)

def run_annotation_tool(images_dir, json_dir):
    json_files = [f for f in os.listdir(json_dir) if f.endswith('.json')]
    for i, file in enumerate(json_files):
        img_path = os.path.join(images_dir, file.replace('.json', '.jpg'))
        json_path = os.path.join(json_dir, file)
        if os.path.exists(img_path):
            draw_and_relabel(img_path, json_path, "Dataset", i+1, len(json_files))

# --- UNCOMMENT TO RUN TOOL (Only works locally) ---
# run_annotation_tool("path/to/images", "path/to/json")

---
## 3. Format Conversion: LabelMe to YOLO
Once annotations are verified, we convert the `JSON` polygons into YOLO-format `.txt` files. This step normalizes coordinates (0-1) and maps string labels (e.g., "Class 1") to integer IDs (0).


In [ ]:
def convert_labelme_to_yolo(json_dir, output_dir, class_map, img_w=1920, img_h=1080):
    os.makedirs(output_dir, exist_ok=True)
    
    for json_file in os.listdir(json_dir):
        if not json_file.endswith(".json"): continue
        
        with open(os.path.join(json_dir, json_file), 'r') as f:
            data = json.load(f)

        txt_name = os.path.splitext(json_file)[0] + ".txt"
        
        with open(os.path.join(output_dir, txt_name), 'w') as out_f:
            for shape in data.get("shapes", []):
                label = shape.get("label")
                points = shape.get("points", [])
                
                if label not in class_map or not points: continue
                
                class_id = class_map[label]
                
                # Normalize points
                norm_points = []
                for x, y in points:
                    norm_points.append(f"{min(max(x/img_w, 0), 1):.6f}")
                    norm_points.append(f"{min(max(y/img_h, 0), 1):.6f}")
                
                out_f.write(f"{class_id} {' '.join(norm_points)}\n")

# Configuration
YOLO_CLASS_MAP = {
    "Class 1": 0, "Class 2": 1, "Class 3": 2,
    "Class 4": 3, "Class 5": 4, "Class 6": 5
}

# --- EXAMPLE USAGE ---
# convert_labelme_to_yolo(
#     "path/to/json_annotations", 
#     "path/to/yolo_labels", 
#     YOLO_CLASS_MAP
# )

---
## 4. Dataset Splitting
To ensure robust evaluation, we split the dataset into three subsets:

* **Train (70%):** For model learning.
* **Validation (20%):** For hyperparameter tuning during training.
* **Test (10%):** For final unseen evaluation.


In [ ]:
def split_and_organize_dataset(source_images, source_labels, output_dir, split_ratios=(0.7, 0.2, 0.1)):
    # Collect pairs
    pairs = []
    for f in os.listdir(source_images):
        if f.endswith('.jpg'):
            name = os.path.splitext(f)[0]
            label_file = os.path.join(source_labels, name + '.txt')
            if os.path.exists(label_file):
                pairs.append((os.path.join(source_images, f), label_file))

    # Shuffle
    random.shuffle(pairs)
    
    # Calculate indices
    n = len(pairs)
    train_end = int(n * split_ratios[0])
    val_end = train_end + int(n * split_ratios[1])
    
    splits = {
        'train': pairs[:train_end],
        'valid': pairs[train_end:val_end],
        'test': pairs[val_end:]
    }

    # Copy files
    for split_name, split_pairs in splits.items():
        img_dir = os.path.join(output_dir, split_name, 'images')
        lbl_dir = os.path.join(output_dir, split_name, 'labels')
        os.makedirs(img_dir, exist_ok=True)
        os.makedirs(lbl_dir, exist_ok=True)
        
        for i, (img, lbl) in enumerate(tqdm(split_pairs, desc=f"Organizing {split_name}")):
            ext = os.path.splitext(img)[1]
            new_name = f"{split_name}_{i:05d}"
            shutil.copy(img, os.path.join(img_dir, new_name + ext))
            shutil.copy(lbl, os.path.join(lbl_dir, new_name + ".txt"))

# --- EXAMPLE USAGE ---
# split_and_organize_dataset("path/to/all_images", "path/to/all_labels", "D:/JKR_Data_Final")

---
## 5. Data Augmentation
To improve model generalization and address the limited data size, we apply offline augmentation. We generate synthetic variations using:

* **Horizontal Flip:** Simulates different camera angles.
* **Vertical Flip:** Adds geometric variety.
* **Grayscale:** Encourages the model to learn shape features rather than relying solely on color.


In [ ]:
def load_yolo_labels(path, w, h):
    with open(path, 'r') as f: lines = f.readlines()
    polys = []
    for line in lines:
        parts = line.strip().split()
        cls = parts[0]
        coords = [float(x) for x in parts[1:]]
        # Denormalize
        points = [(coords[i]*w, coords[i+1]*h) for i in range(0, len(coords), 2)]
        polys.append((cls, points))
    return polys

def save_aug_labels(path, polys, w, h):
    with open(path, 'w') as f:
        for cls, pts in polys:
            norm = [f"{x/w:.6f} {y/h:.6f}" for x, y in pts]
            f.write(f"{cls} {' '.join(norm)}\n")

def run_augmentation(dataset_path):
    for split in ['train', 'valid', 'test']:
        img_dir = os.path.join(dataset_path, split, 'images')
        lbl_dir = os.path.join(dataset_path, split, 'labels')
        
        files = [f for f in os.listdir(img_dir) if f.endswith('.jpg')]
        
        transform_h = A.Compose([A.HorizontalFlip(p=1)], keypoint_params=A.KeypointParams(format='xy'))
        transform_v = A.Compose([A.VerticalFlip(p=1)], keypoint_params=A.KeypointParams(format='xy'))
        transform_g = A.Compose([A.ToGray(p=1)]) # Grayscale doesn't affect keypoints
        
        for file in tqdm(files, desc=f"Augmenting {split}"):
            img_path = os.path.join(img_dir, file)
            lbl_path = os.path.join(lbl_dir, file.replace('.jpg', '.txt'))
            
            if not os.path.exists(lbl_path): continue
            
            image = cv2.imread(img_path)
            h, w = image.shape[:2]
            polys = load_yolo_labels(lbl_path, w, h)
            
            # Helper to process transform
            def apply_trans(name, transform, affects_coords=True):
                # Prepare Keypoints (flatten list of all polygon points)
                all_pts = []
                poly_map = [] # To reconstruct polygons later
                for cls, pts in polys:
                    poly_map.append(len(pts))
                    all_pts.extend(pts)
                
                if affects_coords:
                    augmented = transform(image=image, keypoints=all_pts)
                    aug_img = augmented['image']
                    aug_kps = augmented['keypoints']
                else:
                    augmented = transform(image=image)
                    aug_img = augmented['image']
                    aug_kps = all_pts # Points don't change for grayscale
                
                # Save Image
                cv2.imwrite(os.path.join(img_dir, f"{os.path.splitext(file)[0]}_{name}.jpg"), aug_img)
                
                # Reconstruct and Save Label
                new_polys = []
                idx = 0
                for i, count in enumerate(poly_map):
                    cls = polys[i][0]
                    new_pts = aug_kps[idx : idx+count]
                    new_polys.append((cls, new_pts))
                    idx += count
                
                save_aug_labels(os.path.join(lbl_dir, f"{os.path.splitext(file)[0]}_{name}.txt"), new_polys, w, h)

            # Apply
            apply_trans("flip_h", transform_h)
            apply_trans("flip_v", transform_v)
            apply_trans("gray", transform_g, affects_coords=False)

# --- UNCOMMENT TO RUN ---
# run_augmentation("D:/JKR_Data_Final")

---
## 6. Model Training (YOLOv11)
Finally, we train the YOLOv11 model using the **Ultralytics** framework.

* **Task:** Segmentation (Instance Segmentation) or Detection.
    * *Note: The polygon-shaped annotations are suitable for both object detection and instance segmentation tasks*
* **Epochs:** 50-100 (Early stopping enabled).
* **Image Size:** 640 or 1080.

In [ ]:
# Create data.yaml
yaml_content = f"""
path: D:/JKR_Data_Final  # Dataset root dir
train: train/images
val: valid/images
test: test/images

# Classes
names:
  0: Class 1
  1: Class 2
  2: Class 3
  3: Class 4
  4: Class 5
  5: Class 6
"""

with open("myvid_data.yaml", "w") as f:
    f.write(yaml_content)

print("✅ data.yaml created.")

In [ ]:
def train_model():
    # Load a pre-trained YOLOv11 segmentation model
    # Use 'yolo11n-seg.pt' for nano (fastest) or 'yolo11s-seg.pt' for small (better accuracy)
    model = YOLO('yolo11s-seg.pt') 

    results = model.train(
        data='myvid_data.yaml',
        epochs=100,
        imgsz=640,
        batch=16,
        patience=15,    # Early stopping
        device=0,       # Use GPU (0) or CPU ('cpu')
        project='MYVID_Training',
        name='yolo11s_run1',
        verbose=True
    )
    
    print("🚀 Training Complete!")
    return model

# --- UNCOMMENT TO START TRAINING ---
# model = train_model()

---
## 7. Conclusion
This notebook successfully demonstrates the complete data pipeline for the MY-VID dataset. From processing raw video frames to training a sophisticated YOLOv11 model, every step is automated and reproducible. The resulting model weights (best.pt) are saved in the MYVID_Training directory and are ready for deployment in the TrafficSense AI GUI.